# Rozdział 7: Budowanie aplikacji czatujących
## Szybki start z OpenAI API

Ten notatnik jest adaptacją z [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst), który zawiera notatniki korzystające z usług [Azure OpenAI](notebook-azure-openai.ipynb).

Python OpenAI API działa również z modelami Azure OpenAI, z kilkoma modyfikacjami. Dowiedz się więcej o różnicach tutaj: [Jak przełączać się między punktami końcowymi OpenAI i Azure OpenAI za pomocą Pythona](https://learn.microsoft.com/azure/ai-services/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Przegląd  
„Duże modele językowe to funkcje, które odwzorowują tekst na tekst. Mając ciąg wejściowy tekstu, duży model językowy próbuje przewidzieć, jaki tekst pojawi się następny” (1). Ten notatnik „quickstart” wprowadzi użytkowników w podstawowe koncepcje LLM, podstawowe wymagania pakietu do rozpoczęcia pracy z AML, łagodne wprowadzenie do projektowania promptów oraz kilka krótkich przykładów różnych zastosowań. 


## Spis treści  

[Przegląd](#overview)  
[Jak korzystać z usługi OpenAI](#how-to-use-openai-service)  
[1. Tworzenie usługi OpenAI](#1.-creating-your-openai-service)  
[2. Instalacja](#2.-installation)    
[3. Dane uwierzytelniające](#3.-credentials)  

[Przypadki użycia](#use-cases)    
[1. Podsumowanie tekstu](#1.-summarize-text)  
[2. Klasyfikacja tekstu](#2.-classify-text)  
[3. Generowanie nowych nazw produktów](#3.-generate-new-product-names)  
[4. Dostosowanie klasyfikatora](#4.fine-tune-a-classifier)  

[Bibliografia](#references)


### Stwórz swój pierwszy prompt  
To krótkie ćwiczenie zapewni podstawowe wprowadzenie do przesyłania promptów do modelu OpenAI w celu wykonania prostego zadania „streszczenie”.  


**Kroki**:  
1. Zainstaluj bibliotekę OpenAI w swoim środowisku Pythona  
2. Załaduj standardowe biblioteki pomocnicze i ustaw typowe dane uwierzytelniające bezpieczeństwa OpenAI dla utworzonej przez siebie usługi OpenAI  
3. Wybierz model do swojego zadania  
4. Stwórz prosty prompt dla modelu  
5. Wyślij swoje żądanie do API modelu!  


### 1. Zainstaluj OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Importuj biblioteki pomocnicze i utwórz instancję poświadczeń


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Znajdowanie odpowiedniego modelu  
Modele takie jak GPT-4o i GPT-4o mini potrafią rozumieć i generować język naturalny, i są dostępne na platformie OpenAI z różnymi poziomami mocy i szybkości odpowiednimi do różnych zadań.


In [ ]:
# Select a general purpose chat model
model = "gpt-4o-mini"


## 4. Projektowanie promptu  

"Magia dużych modeli językowych polega na tym, że dzięki trenowaniu ich do minimalizowania błędu predykcji na ogromnych ilościach tekstu, modele uczą się pojęć przydatnych do tych przewidywań. Na przykład uczą się pojęć takich jak"(1):

* jak się pisze
* jak działa gramatyka
* jak parafrazować
* jak odpowiadać na pytania
* jak prowadzić rozmowę
* jak pisać w wielu językach
* jak programować
* itp.

#### Jak kontrolować duży model językowy  
"Spośród wszystkich wejść do dużego modelu językowego zdecydowanie najbardziej wpływowy jest prompt tekstowy(1).

Duże modele językowe mogą być stymulowane do generowania wyników na kilka sposobów:

Instrukcja: Powiedz modelowi, czego chcesz
Uzupełnienie: Skłonić model do dokończenia początku tego, czego chcesz
Demonstracja: Pokaż modelowi, czego chcesz, za pomocą:
Kilku przykładów w promptcie
Setek lub tysięcy przykładów w zbiorze treningowym do dopracowania"



#### Istnieją trzy podstawowe zasady tworzenia promptów:

**Pokaż i powiedz**. Wyraźnie zaznacz, czego chcesz, przez instrukcje, przykłady lub ich kombinację. Jeśli chcesz, aby model uporządkował listę elementów w kolejności alfabetycznej albo sklasyfikował akapit pod kątem sentymentu, pokaż mu, że tego oczekujesz.

**Dostarcz wysokiej jakości dane**. Jeśli próbujesz zbudować klasyfikator lub sprawić, by model podążał za wzorcem, upewnij się, że masz wystarczającą liczbę przykładów. Upewnij się, że poprawiłeś swoje przykłady — model zazwyczaj jest na tyle mądry, by dostrzec podstawowe błędy ortograficzne i udzielić odpowiedzi, ale może też założyć, że jest to celowe i to może wpłynąć na odpowiedź.

**Sprawdź swoje ustawienia.** Ustawienia temperature i top_p kontrolują, jak deterministyczny jest model przy generowaniu odpowiedzi. Jeśli pytasz o odpowiedź, gdzie jest tylko jedna prawidłowa, powinieneś ustawić je na niższe wartości. Jeśli szukasz bardziej zróżnicowanych odpowiedzi, możesz ustawić je wyżej. Najczęstszym błędem jest myślenie, że te ustawienia kontrolują "spryt" lub "kreatywność".


Źródło: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. Prześlij!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Powtórz to samo wywołanie, jak porównują się wyniki?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Podsumuj tekst  
#### Wyzwanie  
Podsumuj tekst, dodając na końcu fragmentu tekstu 'tl;dr:'. Zauważ, jak model rozumie, jak wykonać wiele zadań bez dodatkowych instrukcji. Możesz eksperymentować z bardziej opisowymi podpowiedziami niż tl;dr, aby zmodyfikować zachowanie modelu i dostosować otrzymywane podsumowanie(3).  

Ostatnie prace wykazały znaczne postępy w wielu zadaniach i benchmarkach NLP dzięki wstępnemu treningowi na dużym korpusie tekstu, a następnie dopracowaniu na konkretnym zadaniu. Chociaż architektura jest zwykle niezależna od zadania, metoda ta wciąż wymaga doczepienia specyficznych dla zadania zestawów danych liczących tysiące lub dziesiątki tysięcy przykładów. Dla kontrastu, ludzie zazwyczaj potrafią wykonać nowe zadanie językowe na podstawie zaledwie kilku przykładów lub prostych instrukcji — coś, z czym obecne systemy NLP wciąż mają duże trudności. Tutaj pokazujemy, że skalowanie modeli językowych znacznie poprawia wydajność w zadaniach niezależnych od zadania i wykonywanych na podstawie kilku przykładów, czasem nawet osiągając konkurencyjność względem wcześniejszych, najlepszych metod opartych na dopracowywaniu.  



Tl;dr


# Ćwiczenia dla różnych przypadków użycia  
1. Streszczanie tekstu  
2. Klasyfikacja tekstu  
3. Generowanie nowych nazw produktów


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Klasyfikuj tekst  
#### Zadanie  
Klasyfikuj elementy do kategorii podanych w czasie wnioskowania. W poniższym przykładzie podajemy zarówno kategorie, jak i tekst do sklasyfikowania w prompt(*playground_reference). 

Zapytanie klienta: Witam, jeden z klawiszy na klawiaturze mojego laptopa niedawno się zepsuł i potrzebuję wymiany:

Zaklasyfikowana kategoria:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Generowanie nowych nazw produktów
#### Wyzwanie
Tworzenie nazw produktów na podstawie przykładowych słów. Tutaj wprowadzamy do promptu informacje o produkcie, dla którego będziemy generować nazwy. Podajemy również podobny przykład, aby pokazać wzorzec, jaki chcemy otrzymać. Ustawiliśmy także wysoką wartość temperatury, aby zwiększyć losowość i uzyskać bardziej innowacyjne odpowiedzi.

Opis produktu: Domowa maszyna do robienia shake'ów mlecznych
Słowa kluczowe: szybki, zdrowy, kompaktowy.
Nazwy produktów: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Opis produktu: Para butów, które pasują na każdy rozmiar stopy.
Słowa kluczowe: adaptowalny, dopasowany, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Odniesienia  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Najlepsze praktyki dostrajania modeli GPT do klasyfikacji tekstu](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Po więcej pomocy  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# Współtwórcy
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
